# MusicGen Dreamboothing on Kaggle

This notebook is optimized for running on Kaggle with T4 GPUs (multi-GPU supported).

In [ ]:
%%capture
# 1. Install dependencies
!git clone https://github.com/Phan-Trung-Thuan/musicgen-dreamboothing
%cd musicgen-dreamboothing
!pip install -r requirements-kaggle.txt
!pip install -e .

**Restart the Kernel (Run -> Restart) after installation, then continue below.**

In [ ]:
# 2. Prepare Metadata
%cd /kaggle/working/musicgen-dreamboothing
# Update these paths to your Kaggle input
SOURCE_DIR = "/kaggle/input/datasets/phantrungthuan/vietnamese-music-dataset"
CAPTION_FILE = "/kaggle/input/datasets/phantrungthuan/vietnamese-music-dataset/vietnamese-music-captioning.json"
DEST_DIR = "."
TEST_SPLIT = 0.2 # 20% for testing

!python prepare_kaggle_dataset.py \
    --source_dir "{SOURCE_DIR}" \
    --caption_file "{CAPTION_FILE}" \
    --dest_dir "{DEST_DIR}" \
    --test_split {TEST_SPLIT}

In [ ]:
# 3. Preprocess Dataset (RAM Optimization)
# This converts audio to small tokens once to avoid RAM OOM during training.
PROCESSED_DIR = "./processed_dataset"

!python preprocess_dataset.py \
    --model_name_or_path facebook/musicgen-small \
    --dataset_dir . \
    --output_dir {PROCESSED_DIR} \
    --max_duration_in_seconds 30

In [ ]:
# 4. Run Training
import os
OUTPUT_DIR = "./musicgen-vietnamese-lora"

command = (
    f"accelerate launch --multi_gpu --mixed_precision=fp16 dreambooth_musicgen.py "
    f"    --model_name_or_path facebook/musicgen-small "
    f"    --dataset_name {os.path.abspath(PROCESSED_DIR)} "
    f"    --target_audio_column_name audio_path "
    f"    --text_column_name text "
    f"    --output_dir {OUTPUT_DIR} "
    f"    --use_lora "
    f"    --do_train "
    f"    --do_eval "
    f"    --fp16 "
    f"    --num_train_epochs 5 "
    f"    --max_duration_in_seconds 30 "
    f"    --preprocessing_num_workers 4 "
    f"    --dataloader_num_workers 4 "
    f"    --learning_rate 2e-4 "
    f"    --per_device_train_batch_size 1 "
    f"    --per_device_eval_batch_size 1 "
    f"    --gradient_accumulation_steps 8 "
    f"    --gradient_checkpointing True "
    f"    --gradient_checkpointing_kwargs '{{\"use_reentrant\": false}}' "
    f"    --ddp_find_unused_parameters False "
    f"    --evaluation_strategy steps "
    f"    --eval_steps 100 "
    f"    --eval_accumulation_steps 1 "
    f"    --save_steps 100 "
    f"    --save_total_limit 2 "
    f"    --load_best_model_at_end True "
    f"    --metric_for_best_model eval_loss "
    f"    --greater_is_better False "
    f"    --logging_steps 10 "
    f"    --train_split_name train "
    f"    --eval_split_name test "
    f"    --decoder_start_token_id 2047 "
    f"    --pad_token_id 2047"
)

print(f"Running command: {command}")
!{command}

# 5. Inference
Load the fine-tuned LoRA model and generate audio.

In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import IPython.display as ipd

# 1. Load base model and processor
model_id = "facebook/musicgen-small"
processor = AutoProcessor.from_pretrained(model_id)
model = MusicgenForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16)

# 2. Load LoRA weights
model.load_adapter(OUTPUT_DIR)
model.to("cuda")

prompts = ["Vietnamese folk music with traditional instruments"]
inputs = processor(text=prompts, padding=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    # 1500 tokens is approximately 30 seconds of audio
    audio_values = model.generate(**inputs, max_new_tokens=1500)

sampling_rate = model.config.audio_encoder.sampling_rate
for audio in audio_values:
    ipd.display(ipd.Audio(audio[0].cpu().numpy(), rate=sampling_rate))

# 6. Archive Model
Zip the trained model folder for downloading.

In [ ]:
!zip -r musicgen_lora_model.zip {OUTPUT_DIR}